**RunableParallel**

In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [3]:
chat_template_books = ChatPromptTemplate.from_template('''Suggest three of the best intermediate-level {programming language} books
Answer only by listing the book. ''') 

chat_template_projects = ChatPromptTemplate.from_template('''Suggest three interesting {programming language} projects suitable
for intermediate-level programmers. Answer only by listing the projects.''')

chat_template_time = ChatPromptTemplate.from_template(
    '''
    I'm an intermediate level programmer. Consider the following literature: {books}

    Also, consider the following projects: {projects}

    Roughly how much time would it take me to complete the literature and projects?
    '''
)

In [4]:
chat = ChatOpenAI(model_name = 'gpt-4',
                  model_kwargs = {'seed':365},
                  temperature = 0,
                  max_tokens = 100)

/home/myxubuntuos/.local/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3448: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


In [5]:
string_parser = StrOutputParser()

In [6]:
# Passing Chain Tools Output to Chain Strategy 
chain_books = chat_template_books | chat | string_parser   
chain_projects = chat_template_projects | chat | string_parser 

In [7]:
# One line Chain
chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_projects})

In [8]:
chain_parallel.invoke({'programming language':'Python'})

{'books': '1. "Fluent Python: Clear, Concise, and Effective Programming" by Luciano Ramalho\n2. "Python Cookbook: Recipes for Mastering Python 3" by David Beazley and Brian K. Jones\n3. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin',
 'projects': '1. Building a Web Scraper using BeautifulSoup.\n2. Developing a simple Machine Learning model using Scikit-learn.\n3. Creating a GUI application using Tkinter or PyQt.'}

In [17]:
chain_time1 = (RunnableParallel({'books':chain_books, 'projects':chain_projects}) | chat_template_time | chat | string_parser)

In [18]:
chain_time2 = ({'books':chain_books, 'projects':chain_projects} | chat_template_time | chat | string_parser)

In [19]:
print(chain_time1.invoke({'programming language':'Python'}))

The time it would take to complete the literature and projects can vary greatly depending on several factors such as your current skill level, the amount of time you can dedicate each day, your reading speed, and how quickly you grasp new concepts. 

However, as a rough estimate:

1. "Fluent Python: Clear, Concise, and Effective Programming" - This book is around 800 pages. If you read and practice for about 2 hours a day, it might take you around 1


In [20]:
chain_time2.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

In [ ]:
''' Key Takeaway: You Don't Need to Wrap Runnables in a Parallal Class if you call them at the same time.
chain_time2 is better than chain_time1 due to cleaner code.'''